← [Overview](../00_overview.ipynb)

# Averaging

`averaging` is the one **time-based** grouping method in tsam, and the simplest
member of this clustering section. It splits the ordered period list into $k$
**consecutive equal-size blocks** — purely by temporal position, without looking at the
values at all — and represents each block by its mean.

That makes it the odd one out among the clustering methods covered here: the
[partitional](01_partitional_clustering.ipynb),
[agglomerative](02_agglomerative_clustering.ipynb) and
[extremal-prototype](03_extremal_prototype_selection.ipynb) methods are all
**feature-based** (they group by value similarity), whereas averaging is
**time-based**. It still fits the same pipeline: it produces a cluster
assignment and one representative per cluster, exactly like the others.


In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"

# --------------------------------------------------------------------------
# The shared tiny six-day dataset introduced in 01_preprocessing and reused
# across this clustering section (see 01_partitional_clustering for the
# `day_p` shape legend). Averaging is traced on the same six days so it can be
# compared directly with the feature-based methods.
# --------------------------------------------------------------------------
tiny = pd.read_csv("../../../data/tiny.csv", index_col=0, parse_dates=True)
UNITS = {"solar": "W/m²", "load": "MW"}
print("tiny:", tiny.shape)

---

## Mechanism

Given $N$ periods and target $k$ clusters, the assignment is purely positional:

* Block $0$: periods $0 \ldots \lfloor N/k \rfloor - 1$
* Block $1$: periods $\lfloor N/k \rfloor \ldots 2\lfloor N/k \rfloor - 1$
* …
* Any remainder is appended to the last block.

The **cluster representative** is the mean of the block members — hence "averaging".

In [ ]:
k_avg = 3
n_periods = 6
block_size = n_periods // k_avg
remainder = n_periods - block_size * k_avg

print(f"k={k_avg}, N={n_periods}, block_size={block_size}, remainder={remainder}")

avg_assignments = []
for c in range(k_avg):
    avg_assignments.extend([c] * block_size)
if remainder > 0:
    avg_assignments.extend([k_avg - 1] * remainder)

print("\nAveraging assignments (purely positional — values not consulted):")
for i, a in enumerate(avg_assignments):
    print(f"  day_{i} -> cluster {a}")

print(
    "\nNote: days 0 and 1 both happen to be sunny days that land in the same cluster,"
)
print(
    "but this is by position, not similarity. The algorithm has no knowledge of values."
)

### Averaging on the calendar

Because averaging groups by position, the clusters are simply consecutive
equal-size blocks of the calendar — the same block shape as `contiguous`
clustering, but chosen without ever looking at the values.

On this tiny set that is a lucky match: the six days are *already ordered* so
that similar shapes are neighbours (sunny 0–1, overcast 2–3, cloudy 4–5), so the
positional blocks land on exactly the same three pairs the feature-based methods
find. That is a coincidence of ordering, not a property of averaging — scramble
the calendar and the blocks would cut straight across the shape groups. See
[Comparing clustering methods](../../../tutorials/comparing_clustering_methods.ipynb)
for the accuracy gap this opens up on a realistic, unordered series.

In [ ]:
result_avg_tiny.plot.clusters_over_time(
    columns=["load"], units=UNITS, title="Averaging: consecutive positional blocks"
)

---

## Beyond positional blocks: supply your own assignment vector

`averaging` groups strictly by position. For a **calendar-based** grouping —
two-day blocks, week-of-year, season x weekday, or any rule of your own — you
build the assignment vector yourself (one cluster id per period) and hand it to
tsam through `ClusteringResult.apply()`, which **reuses a given
`cluster_assignments`** instead of running a clustering algorithm. With
`representation="mean"` each group is still averaged, so this is *averaging by
your own grouping*.

Below we group the six days into **two-day blocks** and read off the mean
representatives. Here that happens to coincide with what `averaging` itself
produces; on a longer series the same call handles groupings averaging cannot,
such as week-of-year or season x weekday.

In [ ]:
from tsam import ClusteringResult

# Build the grouping yourself: one cluster id per period (day).
# Two-day blocks -> [0, 0, 1, 1, 2, 2]. Any calendar rule is just a different
# formula for this vector (e.g. week-of-year: ((days - days[0]).days // 7)).
days = tiny.index[::4]  # one timestamp per period
two_day_id = np.arange(len(days)) // 2  # 0, 0, 1, 1, 2, 2

clustering = ClusteringResult(
    period_duration=24.0,  # hours per period (1 day)
    n_timesteps_per_period=4,  # four 6-hourly steps per day
    cluster_assignments=tuple(int(c) for c in two_day_id),
    representation="mean",  # each block -> its average
    temporal_resolution=6.0,  # hours per timestep
)
result_ext = clustering.apply(tiny)

print("external-vector assignments:", result_ext.cluster_assignments)
print("cluster counts:", result_ext.cluster_counts)
print("\nTwo-day block representatives (mean profile per block):")
result_ext.cluster_representatives

---

**Up next:**
* [Representation](../03_representation.ipynb) — how each cluster (from any of these grouping methods) becomes a single profile

**See also:**
* [Comparing clustering methods](../../../tutorials/comparing_clustering_methods.ipynb) — averaging vs the feature-based methods on a realistic series